# Multimodal Misinformation Detection — full pipeline

This notebook runs the entire dissertation artefact stage by stage, in the
same order as the `Makefile`. Run the cells from top to bottom with
**Shift+Enter**, or use **Run → Run All Cells** to run everything unattended.

Every stage caches its output to disk, so if a cell fails or you stop partway
through, re-running from the top skips whatever is already done rather than
redoing it.

**Before you start:** this notebook must be running the kernel from this
project's own virtual environment (`.venv`), not your Mac's system Python —
check the kernel name in the top-right corner of this window. If it doesn't
say something like *Python (mmid-artefact)*, see the setup guide you were
given for how to install and select it, otherwise the imports below will
fail.


In [1]:
# Sanity check: confirm this notebook is running inside the project folder
# and using the project's own virtual environment, not the system Python.
import sys, os
from pathlib import Path

print("Python:      ", sys.executable)
print("Working dir: ", os.getcwd())

if not (Path("src") / "config.py").exists():
    raise SystemExit(
        "This notebook is not running from the mmid-artefact folder.\n"
        "In Jupyter: File > Open... and open run_pipeline.ipynb from inside "
        "mmid-artefact, or run  os.chdir('path/to/mmid-artefact')  in a cell "
        "above this one, then re-run this cell."
    )
print("[ok] found src/config.py -- working directory is correct")


Python:       /Users/prasannakumar/Downloads/Multimodal Misinformation Detection/mmid-artefact/.venv/bin/python
Working dir:  /Users/prasannakumar/Downloads/Multimodal Misinformation Detection/mmid-artefact
[ok] found src/config.py -- working directory is correct


In [ ]:
# Confirm the heavy dependencies are importable and see which compute
# backend PyTorch will use on this Mac (MPS = Apple Silicon acceleration).
import torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("MPS available:", getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())


---
## Stage 1 — build the corpus

Downloads the Fakeddit metadata table, samples a class-balanced corpus capped
at 1,600 posts per subreddit, and fetches the Reddit preview images. Needs an
internet connection. Roughly 15 minutes; safe to re-run if it's interrupted,
it will not re-download what already succeeded.


In [ ]:
!python -m src.build_dataset

---
## Stage 2 — frozen representations (CLIP + DistilRoBERTa)

Encodes every post with the frozen CLIP and DistilRoBERTa models. This is the
first stage that downloads model weights from Hugging Face the first time it
runs (a few hundred MB), then reuses the local cache after that.


In [ ]:
!python -m src.features --stages base

---
## Stage 2b — language-model features (optional, slower)

Adds Qwen2.5-0.5B-Instruct hidden states and its zero-shot judgement. Needed
for the `+LLM` model variants and for the report's proposed system
(CGF + LLM + adversarial). Skip this cell if you only want the faster core
result — every later stage still runs without it, using plain CGF instead.


In [ ]:
!python -m src.features --stages llm

---
## Stage 4 — the experiment grid

Trains every model variant under all three evaluation protocols and three
seeds, and caches the test-set predictions. This is the longest stage on CPU.


In [ ]:
!python -m src.train

---
## Stage 6 — robustness and behavioural testing

Text perturbations, the out-of-context probe, and measured inference cost.


In [ ]:
!python -m src.robustness

---
## Stage 5 — aggregation, significance tests and figures

Reads everything cached so far and writes the tables and all eight report
figures to `figures/`.


In [ ]:
!python -m src.analyse

---
## Stages 10–12 — the extended analyses

The leave-one-community-out rotation, the permutation null for the shortcut
probe, per-community and ranking-quality results, the prior-shift
decomposition, and the Holm-Bonferroni correction; then the reviewer triage
queue.


In [ ]:
!python -m src.extra_analyses

In [ ]:
!python -m src.triage

---
## Stage 13 — flowcharts (optional)

Regenerates the `.drawio` diagram sources and their rendered figures. Not
needed to reproduce any numeric result; only run this if you've edited
`src/flowcharts.py`.


In [ ]:
!python -m src.flowcharts

---
## Verify everything

Runs the test suite: leakage checks, split disjointness, data minimisation,
metric correctness and model shape/determinism tests.


In [ ]:
!python -m pytest tests/ -q

---
## Score one image and headline

Once Stage 2 (and, optionally, 2b) has run at least once, you can check any
image/headline pair interactively — the same thing `python -m src.predict`
does from the command line, but inline here so you can see the result and
iterate on the headline without leaving the notebook.

Edit `IMAGE_PATH` and `HEADLINE` below and re-run the cell.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == "src" else str(Path.cwd()))
from src.predict import score, explain

IMAGE_PATH = "sandisk-pAKgXLu04CQ-unsplash.jpg"   # change to any image path
HEADLINE = "Scientists discover water on the surface of the sun"

result = score(IMAGE_PATH, HEADLINE)
print(explain(result))


---
## Rebuild the Word report (optional)

Needs `pandoc` installed (`brew install pandoc`). Regenerates
`docs/report.docx` from `docs/report.md` and `docs/appendices.md` with every
table and figure re-pulled from the results just produced above.


In [ ]:
%%bash
cd docs && pandoc report.md appendices.md -o report.docx --toc --toc-depth=2 --resource-path=.:..
cd .. && python -m src.style_docx "docs/report.docx"
